<a href="https://colab.research.google.com/github/Innovatewithapple/TransformersProjects/blob/main/BERTCustomModelPytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install transformers

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset

In [8]:
autotoken = AutoTokenizer.from_pretrained('bert-base-uncased')
autotoken.vocab_size

30522

In [ ]:
raw_datasets = load_dataset('imdb')

train_Dataset = raw_datasets['train']
val_Dataset = raw_datasets['test']

In [19]:
embed_dim=256
vocab_size=autotoken.vocab_size
max_len=200
num_heads=8
ff_dim=128
num_layers=8
num_classes=2

In [33]:
class IMDBDataset(Dataset):
  def __init__(self,text,labels,max_len):
    self.text = text
    self.tokenizer = autotoken
    self.lables = labels
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self,idx):
    # clean the text, real world has messy or nan
    item = str(self.text[idx])

    encoding = self.tokenizer(text=item,padding='max_length',max_length=self.max_len,add_special_tokens=True,truncation=True,return_attention_mask=True,return_tensors='pt')

    return {
        'input_ids':encoding['input_ids'].flatten(),
        'attention_mask': encoding['attention_mask'].flatten(),
        'labels': torch.tensor(self.lables[idx],dtype=torch.long)
    }


In [37]:
train_data = IMDBDataset(text=train_Dataset['text'],labels=train_Dataset['label'],max_len=max_len)
val_data = IMDBDataset(text=val_Dataset['text'],labels=val_Dataset['label'],max_len=max_len)

In [38]:
train_loader = DataLoader(dataset=train_data,batch_size=32,shuffle=True,pin_memory=True)
val_loader = DataLoader(dataset=val_data,batch_size=32,shuffle=False,pin_memory=True)

In [15]:
class TransformerBlock(nn.Module):
  def __init__(self,embed_dim,num_heads,ff_dim) -> None:
    super().__init__()
    self.attention = nn.MultiheadAttention(embed_dim=embed_dim,num_heads=num_heads)

    self.mlp = nn.Sequential(
        nn.Linear(in_features=embed_dim,out_features=ff_dim),
        nn.GELU(),
        nn.Linear(in_features=ff_dim,out_features=embed_dim),
        nn.Dropout(0.2)
    )

    self.layernorm1 = nn.LayerNorm(normalized_shape=embed_dim)
    self.layernorm2 = nn.LayerNorm(normalized_shape=embed_dim)

  def forward(self,input):
    att,_ = self.attention(input,input,input)
    merged_att = self.layernorm1(input + att)
    mlp_output = self.mlp(merged_att)
    return self.layernorm2(merged_att + mlp_output)


In [22]:
class WordEmbedding(nn.Module):
  def __init__(self,embed_dim,vocab_size) -> None:
    super().__init__()

    self.wordEmbed = nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim)

  def forward(self,input):
    return self.wordEmbed(input)

In [1]:
class NLPTransformer(nn.Module):
  def __init__(self,embed_dim,vocab_size,max_len,num_heads,ff_dim,num_layers,num_classes) -> None:
    super().__init__()
    self.wordEmbed = WordEmbedding(embed_dim=embed_dim,vocab_size=vocab_size)
    self.positionEmbed = nn.Parameter(torch.zeros(1,max_len,embed_dim))

    self.transform_layers = nn.ModuleList([
        TransformerBlock(embed_dim=embed_dim,num_heads=num_heads,ff_dim=ff_dim)
        for _ in range(num_layers)
    ])

    self.output_layer = nn.Linear(in_features=embed_dim,out_features=num_classes)

  def forward(self,input,mask):
    x = self.wordEmbed(input) + self.positionEmbed

    for transformer_layer in self.transform_layers:
      x = transformer_layer(x)

    # 3. THE SMART MEAN (Replacing x = x.mean(dim=1))
    # We need to make the mask (Batch, 512) match x (Batch, 512, 128)
    mask = mask.unsqueeze(-1) # shape becomes [Batch, 512, 1]
    x = x * mask
    # Sum only the real words and divide by the count of real words
    x = x.sum(dim=1) / mask.sum(dim=1)
    return self.output_layer(x)

NameError: name 'nn' is not defined

In [20]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [24]:
model = NLPTransformer(embed_dim=embed_dim,
                       vocab_size=vocab_size,
                       max_len=max_len,
                       num_heads=num_heads,
                       ff_dim=ff_dim,
                       num_layers=num_layers,
                       num_classes=num_classes).to(device)

In [25]:
optimizer = optim.Adam(params=model.parameters(),lr=0.001)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
epochs = 5

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for batch in train_loader:
     # 1. Unpack all three items from your Dictionary
    input_ids = batch['input_ids'].to(device)
    mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()

    output = model(input)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    train_loss += loss.item()
    pred = torch.argmax(input=output,dim=1)
    train_correct += (pred == labels).sum().item()
    train_total += labels.size(0)

  train_accuracy = train_correct / train_total
  train_losses = train_loss / len(train_loader)

  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for input,labels in val_loader:
     input = input.to(device)
     labels = labels.to(device)

     optimizer.zero_grad()

     output = model(input)
     loss = loss_fn(output,labels)

     loss.forward()
     optimizer.step()

     val_loss += loss.item()
     pred = torch.argmax(input=output,dim=1)
     val_correct += (pred == labels).sum().item()
     val_total += labels.size(0)

    val_accuracy = val_correct / val_total
    val_losses = val_loss / len(val_loader)

    print(f'\nEpochs: {epoch+1}/{epochs}...')
    print(f'\nTrain_acc: {train_accuracy} | Train_loss: {train_losses}')
    print(f'\nVal_acc: {val_accuracy} | Val_loss: {val_losses}')
